# YOLO11 Pothole Training on Google Colab

This notebook trains the project's YOLO11s detector using the `dek_Engine` GitHub branch. It does not commit model weights to GitHub. Use a Colab GPU runtime before running the training cell.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics

## 2. Clone the project and select the training data

The repository is cloned directly in Colab, so the dataset and corrected `data/v1/data.yaml` are available to the training process.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/moha6959/SemProject_MLEngine.git"
BRANCH = "dek_Engine"
ROOT = Path("/content/SemProject_MLEngine")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        check=True,
    )

os.chdir(ROOT)
DATA_YAML = ROOT / "data/v1/data.yaml"
print("Project root:", ROOT)
print("Dataset config:", DATA_YAML)
print("Dataset config exists:", DATA_YAML.exists())

## 3. Verify the dataset

This check prevents training from starting if the GitHub clone is missing image or label files.

In [ ]:
assert DATA_YAML.is_file(), f"Missing dataset config: {DATA_YAML}"

splits = {
    "train": ROOT / "data/v1/train",
    "valid": ROOT / "data/v1/valid",
    "test": ROOT / "data/v1/test",
}

for split, directory in splits.items():
    images = list((directory / "images").glob("*"))
    labels = list((directory / "labels").glob("*.txt"))
    assert images, f"No images found for {split}: {directory / 'images'}"
    assert labels, f"No labels found for {split}: {directory / 'labels'}"
    print(f"{split}: {len(images)} images, {len(labels)} labels")

print(DATA_YAML.read_text())

## 4. Confirm the Colab runtime

For practical training, use **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch

HAS_GPU = torch.cuda.is_available()
DEVICE = 0 if HAS_GPU else "cpu"
BATCH = -1 if HAS_GPU else 4

print("GPU available:", HAS_GPU)
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: training on CPU will be considerably slower.")

## 5. Train YOLO11s

This is the only cell that starts training. It fine-tunes pretrained YOLO11s on the pothole dataset.

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolo11s.pt"
RUNS_DIR = ROOT / "runs"
RUN_NAME = "pothole_yolo11s"

model = YOLO(MODEL_NAME)
train_results = model.train(
    data="data/v1/data.yaml",
    epochs=100,
    imgsz=640,
    batch=BATCH,
    patience=20,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
)

BEST_WEIGHTS = RUNS_DIR / RUN_NAME / "weights/best.pt"
print("Best weights:", BEST_WEIGHTS)

## 6. Evaluate on the held-out test split

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=DEVICE,
)
print("mAP50:", test_metrics.box.map50)
print("mAP50-95:", test_metrics.box.map)

## 7. Save the trained weights

Model weights are not pushed to GitHub. Save them to Google Drive or download them to your computer for use in VS Code.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/SemProject_MLEngine/models")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
drive_weights = DRIVE_DIR / "pothole_yolo11s_best.pt"
drive_weights.write_bytes(BEST_WEIGHTS.read_bytes())
print("Saved to:", drive_weights)

In [ ]:
from google.colab import files
files.download(str(BEST_WEIGHTS))